# 02 — DQN Training

Train a **Double DQN** agent on the ViZDoom *Basic* scenario. We log learning
curves (reward, loss, epsilon) and render the trained agent's gameplay.

## 1. Setup

In [ ]:
# Uncomment on Colab
# !pip install vizdoom gymnasium torch opencv-python matplotlib tensorboard

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import torch
from collections import deque

from rl_doom.env import DoomEnv, ResizeObservation, SkipFrame, FrameStack
from rl_doom.models import DQNNetwork
from rl_doom.agents.dqn import DQNAgent
from rl_doom.replay_buffer import ReplayBuffer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the lines below when running on Google Colab
# import os
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     os.makedirs(f"{DRIVE_ROOT}/{subdir}", exist_ok=True)
# # Symlink so relative paths (../checkpoints, ../logs, etc.) resolve to Drive
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     local = os.path.abspath(f"../{subdir}")
#     if not os.path.exists(local):
#         os.symlink(f"{DRIVE_ROOT}/{subdir}", local)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")

## 2. Environment

In [ ]:
def make_env(scenario="basic", seed=42):
    env = DoomEnv(scenario=scenario)
    env = ResizeObservation(env, shape=(84, 84))
    env = SkipFrame(env, skip=4)
    env = FrameStack(env, num_stack=4)
    return env

env = make_env()
n_actions = env.action_space.n
obs, _ = env.reset(seed=42)
obs_shape = obs.shape  # Store shape before env gets closed later
print(f"Observation shape: {obs_shape}, Actions: {n_actions}")

## 3. Hyperparameters

In [ ]:
config = dict(
    # Training
    total_steps=100_000,
    learning_starts=1_000,
    train_freq=4,
    batch_size=32,
    # DQN
    lr=1e-4,
    gamma=0.99,
    target_update_freq=1_000,
    # Exploration
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_steps=50_000,
    # Replay
    buffer_size=50_000,
    # Logging
    log_freq=500,
    eval_freq=5_000,
    eval_episodes=10,
)
config

## 4. Initialize agent & replay buffer

In [ ]:
agent = DQNAgent(
    obs_shape=obs_shape,
    n_actions=n_actions,
    lr=config["lr"],
    gamma=config["gamma"],
    device=device,
)

replay_buffer = ReplayBuffer(capacity=config["buffer_size"])

print(f"Network parameters: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## 5. Training loop

In [ ]:
def linear_schedule(step, start, end, duration):
    """Linearly anneal from start to end over duration steps."""
    frac = min(step / duration, 1.0)
    return start + frac * (end - start)


# Logging containers
episode_rewards_log = []
losses_log = []
epsilons_log = []
eval_rewards_log = []

obs, _ = env.reset(seed=42)
episode_reward = 0.0
episode_count = 0
recent_rewards = deque(maxlen=20)

for step in range(1, config["total_steps"] + 1):
    # Epsilon schedule
    epsilon = linear_schedule(
        step, config["eps_start"], config["eps_end"], config["eps_decay_steps"]
    )

    # Select action
    action = agent.select_action(obs, epsilon=epsilon)

    # Step environment
    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    replay_buffer.push(obs, action, reward, next_obs, done)
    obs = next_obs
    episode_reward += reward

    # Episode bookkeeping
    if done:
        episode_count += 1
        episode_rewards_log.append(episode_reward)
        recent_rewards.append(episode_reward)
        obs, _ = env.reset()
        episode_reward = 0.0

    # Train
    if step >= config["learning_starts"] and step % config["train_freq"] == 0:
        batch = replay_buffer.sample(config["batch_size"])
        loss = agent.train_step(batch)
        losses_log.append(loss)

    # Target network update
    if step % config["target_update_freq"] == 0:
        agent.update_target()

    # Logging
    if step % config["log_freq"] == 0:
        epsilons_log.append(epsilon)
        avg = np.mean(recent_rewards) if recent_rewards else 0
        print(
            f"Step {step:>7,} | Eps {epsilon:.3f} | "
            f"Episodes {episode_count} | Avg reward (20) {avg:.2f}"
        )

    # Periodic evaluation
    if step % config["eval_freq"] == 0:
        eval_env = make_env()
        eval_rews = []
        for _ in range(config["eval_episodes"]):
            eo, _ = eval_env.reset()
            er = 0.0
            d = False
            while not d:
                a = agent.select_action(eo, epsilon=0.0)
                eo, r, term, trunc, _ = eval_env.step(a)
                er += r
                d = term or trunc
            eval_rews.append(er)
        eval_env.close()
        eval_rewards_log.append((step, np.mean(eval_rews), np.std(eval_rews)))
        print(f"  >> Eval: {np.mean(eval_rews):.2f} ± {np.std(eval_rews):.2f}")

env.close()
print("\nTraining complete!")

## 6. Learning curves

In [ ]:
os.makedirs("../figures", exist_ok=True)
os.makedirs("../logs", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- Episode rewards --
axes[0].plot(episode_rewards_log, alpha=0.4, label="Raw")
if len(episode_rewards_log) >= 20:
    smoothed = np.convolve(episode_rewards_log, np.ones(20)/20, mode="valid")
    axes[0].plot(range(19, 19 + len(smoothed)), smoothed, label="MA-20")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Episode Rewards")
axes[0].legend()

# -- Loss --
axes[1].plot(losses_log, alpha=0.3)
if len(losses_log) >= 100:
    smoothed = np.convolve(losses_log, np.ones(100)/100, mode="valid")
    axes[1].plot(range(99, 99 + len(smoothed)), smoothed, color="red")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Huber Loss")
axes[1].set_title("Training Loss")

# -- Epsilon --
axes[2].plot(epsilons_log)
axes[2].set_xlabel(f"Log step (x{config['log_freq']})")
axes[2].set_ylabel("Epsilon")
axes[2].set_title("Exploration Schedule")

plt.tight_layout()
plt.savefig("../figures/02_dqn_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Persist training logs for cross-notebook analysis (notebook 04)
np.savez(
    "../logs/dqn_basic_training.npz",
    episode_rewards=np.array(episode_rewards_log),
    losses=np.array(losses_log),
    epsilons=np.array(epsilons_log),
    eval_rewards=np.array(eval_rewards_log) if eval_rewards_log else np.array([]),
)

## 7. Evaluation rewards over training

In [ ]:
if eval_rewards_log:
    steps, means, stds = zip(*eval_rewards_log)
    means, stds = np.array(means), np.array(stds)
    plt.figure(figsize=(10, 5))
    plt.plot(steps, means, marker="o")
    plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
    plt.xlabel("Training Step")
    plt.ylabel("Eval Reward")
    plt.title("DQN — Evaluation Performance (Basic)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../figures/02_dqn_eval_performance.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Hyperparameter sensitivity

Quick exploration: sweep over learning rates to see the effect on final performance.

In [ ]:
# NOTE: This cell is expensive — set QUICK_SWEEP_STEPS to a small value for a
# fast smoke test, or skip this cell entirely.
QUICK_SWEEP_STEPS = 20_000
LR_CANDIDATES = [3e-4, 1e-4, 3e-5]

sweep_results = {}

for lr in LR_CANDIDATES:
    sweep_env = make_env()
    sweep_obs, _ = sweep_env.reset(seed=42)
    sweep_agent = DQNAgent(
        obs_shape=obs_shape, n_actions=n_actions, lr=lr, gamma=0.99, device=device
    )
    sweep_buf = ReplayBuffer(capacity=config["buffer_size"])

    ep_rews, ep_r = [], 0.0

    for s in range(1, QUICK_SWEEP_STEPS + 1):
        eps = linear_schedule(s, 1.0, 0.01, QUICK_SWEEP_STEPS // 2)
        a = sweep_agent.select_action(sweep_obs, epsilon=eps)
        ns, r, term, trunc, _ = sweep_env.step(a)
        sweep_buf.push(sweep_obs, a, r, ns, term or trunc)
        sweep_obs, ep_r = ns, ep_r + r
        if term or trunc:
            ep_rews.append(ep_r)
            sweep_obs, _ = sweep_env.reset()
            ep_r = 0.0
        if s >= 1_000 and s % 4 == 0:
            sweep_agent.train_step(sweep_buf.sample(32))
        if s % 1_000 == 0:
            sweep_agent.update_target()

    sweep_env.close()
    sweep_results[lr] = ep_rews
    print(f"lr={lr:.0e}  episodes={len(ep_rews)}  mean={np.mean(ep_rews[-20:]):.2f}")

# Plot
plt.figure(figsize=(10, 5))
for lr, rews in sweep_results.items():
    if len(rews) >= 10:
        sm = np.convolve(rews, np.ones(10)/10, mode="valid")
        plt.plot(sm, label=f"lr={lr:.0e}")
plt.xlabel("Episode")
plt.ylabel("Reward (MA-10)")
plt.title("LR Sensitivity — DQN Basic")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/02_dqn_lr_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Render trained agent

In [ ]:
from rl_doom.evaluate import record_episode

frames = record_episode(agent, make_env, epsilon=0.0)
print(f"Recorded {len(frames)} frames")

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(6, 6))
ax.axis("off")
img = ax.imshow(frames[0])

def update(i):
    img.set_data(frames[i])
    return [img]

anim = FuncAnimation(fig, update, frames=len(frames), interval=50, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 10. Save checkpoint

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
agent.save("../checkpoints/dqn_basic.pt")
print("Checkpoint saved.")